# 020 — Training the base architectures

Trains `unet`, `resunet`, `attention_unet`, `unet_residual` the four deterministic models.  
Settings:  
- Architecture: Fully-convolutional encoder-decoder
- Weight initialization: He1
- Activation Function (hidden Layers): ReLU
- Activation Function (output): Sigmoid
- Optimizer: Adam
- Batch Size: 8
- Normalization: GroupNorm2
- Loss Function: `0.16*Charbonnier + 0.84*(1- MultiScale-SSIM)`


Imports

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset split
**Split used project-wide:**  
- Real **artworks** are grouped and kept entirely within one fold, exactly as a plain grouped split would do, so no painting leaks across train/val/test.  
- The **mockup** groups (listed in `settings.MOCKUP_ARTWORK_IDS`) exist purely to be learned from. They are split at the individual-pair level, with only `settings.MOCKUP_TEST_RATIO` (default 5%) held out for test. 


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Train the architectures

Each architecture trains in its own **subprocess** (`scripts/train_single.py`).  
Checkpoints go to `models/deterministic/<arch>/best_model.keras`; each subprocess also writes `models/deterministic/<arch>/history.json` so the loss curves below can be plotted after the fact.

In [ ]:
import json
import subprocess

ARCHS = ["resunet", "unet", "attention_unet", "unet_residual"]
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
MODEL_DIR = settings.MODELS_DIR / "deterministic"
LOG_DIR = settings.LOGS_DIR / "deterministic"

histories: dict = {}

for arch in ARCHS:
    cmd = [
        sys.executable,
        "-m",
        "scripts.train_single",
        "--arch",
        arch,
        "--epochs",
        str(EPOCHS),
        "--model-dir",
        str(MODEL_DIR),
        "--log-dir",
        str(LOG_DIR),
    ]
    subprocess.run(cmd, cwd=project_root, check=True)

    history_path = MODEL_DIR / arch / "history.json"
    histories[arch] = json.loads(history_path.read_text())

    best_val_loss = min(histories[arch]["val_loss"])
    print(f"\nBest val_loss ({arch}): {best_val_loss:.4f}")

## 3. Training curves


In [ ]:
for arch, history in histories.items():
    plot_training_curves(history, title=f"Training history — {arch}")
    plt.show()

## 4. Summary


In [ ]:
for arch in ARCHS:
    ckpt = MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")